# Interview Question Generator — GPT API Mini Project

**Use case (Project idea #15):** The user gives a **job role**, and the app returns
**10 tailored interview questions with sample answers**.

**Skill demonstrated:** domain-specific generation + structured output.

**Team:** _(add your names here)_  
**API used:** Groq (free, OpenAI-compatible Chat Completions API)

---
This notebook walks through:
1. Setting up the API key and client
2. A first Chat Completions call
3. Building the core question-generation function
4. Getting clean **structured (JSON)** output we can reuse in an app
5. Experimenting with **parameters** (`temperature`, `max_tokens`, `top_p`) and recording findings
6. Best practices we applied

A companion **Streamlit app** (`interview_question_app.py`) wraps the same logic in a simple UI.

## 1. Setup: API key & client

We use **Groq** — a free, no-credit-card LLM API that is OpenAI-compatible,
so we use the same `openai` Python package and just point it at Groq's endpoint.

Get a free key at **console.groq.com** (Google/GitHub login, no card needed).

Keep the key out of the code in a `.env` file (never commit it). It should contain:

```
GROQ_API_KEY=gsk_your-key-here
```

`load_dotenv()` reads that file into the environment.

In [ ]:
# Groq is OpenAI-compatible, so we use the same openai package
%pip install --upgrade openai python-dotenv -q

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI

# Load variables from the .env file into the environment
load_dotenv()

# Point the OpenAI client at Groq's OpenAI-compatible endpoint
client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

# Quick sanity check that the key was found (prints True / False, never the key itself)
print("API key loaded:", bool(os.getenv("GROQ_API_KEY")))

We will use one model throughout. `gpt-4o-mini` is a good cheap default;
you can swap it for `gpt-3.5-turbo` or `gpt-4o` by changing this one variable.

In [ ]:
MODEL = "llama-3.3-70b-versatile"   # try also: "llama-3.1-8b-instant", "openai/gpt-oss-120b"

## 2. A first Chat Completions call

The Chat Completions API takes a **list of messages**, each with a `role`
(`system`, `user`, or `assistant`) and `content`. The `system` message sets the
assistant's behaviour; the `user` message is the request.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are an experienced technical recruiter."},
        {"role": "user", "content": "Give me one good interview question for a Data Analyst."},
    ],
)

print(response.choices[0].message.content)

## 3. The core function: generate interview questions

We now build a reusable function. The **system prompt** is where we encode all
the behaviour we want — this is the "customized prompt" that turns a generic
chatbot into a focused interview-question tool.

Key design choices in the prompt:
- It must always produce **exactly 10** questions.
- Each question pairs with a **concise sample answer**.
- It should **mix question types** (technical, behavioural, situational) so the
  set feels like a real interview.
- It should adapt difficulty to a **seniority level** the user can choose.

In [ ]:
SYSTEM_PROMPT = """You are an experienced hiring manager and technical recruiter.
Your job is to create realistic interview questions for a given job role.

When given a JOB ROLE (and optionally a seniority level), produce exactly 10
interview questions tailored to that role. For each question, also write a
concise model answer (3-5 sentences) that a strong candidate might give.

Requirements:
- Mix the question types: include technical/role-specific, behavioural
  ("Tell me about a time..."), and situational ("What would you do if...") questions.
- Order questions roughly from warm-up to more challenging.
- Match difficulty to the requested seniority (default to mid-level if none given).
- Keep questions clear and free of bias; do not ask about age, religion,
  marital status, or other protected characteristics.
- Keep sample answers practical and specific to the role, not generic filler."""


def generate_interview_questions(role, seniority="mid-level", **params):
    """Generate interview questions + sample answers for a role.

    Extra keyword args (temperature, max_tokens, top_p, ...) are passed
    straight through to the API so we can experiment with them.
    """
    user_message = f"Job role: {role}\nSeniority level: {seniority}"

    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message},
        ],
        **params,
    )
    return completion.choices[0].message.content

In [ ]:
# Try it out
output = generate_interview_questions("Junior Data Analyst", seniority="entry-level")
print(output)

## 4. Structured output (JSON)

Free-form text is nice to read, but if we want to use the questions in an app
(loop over them, show one at a time, store them), **structured output** is far
easier to work with. We ask the model to return JSON and use
`response_format={"type": "json_object"}` so the reply is guaranteed to be valid JSON.

In [ ]:
JSON_SYSTEM_PROMPT = SYSTEM_PROMPT + """

Return ONLY a valid JSON object with this exact shape:
{
  "role": "<the job role>",
  "seniority": "<the seniority level>",
  "questions": [
    {"type": "technical|behavioural|situational",
     "question": "<the question>",
     "sample_answer": "<a concise model answer>"}
  ]
}
The "questions" array must contain exactly 10 items. Do not add any text outside the JSON."""


def generate_questions_json(role, seniority="mid-level", **params):
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": JSON_SYSTEM_PROMPT},
            {"role": "user", "content": f"Job role: {role}\nSeniority level: {seniority}"},
        ],
        response_format={"type": "json_object"},
        **params,
    )
    return json.loads(completion.choices[0].message.content)

In [ ]:
data = generate_questions_json("Data Analyst", seniority="mid-level")

print("Role:", data["role"], "| Seniority:", data["seniority"])
print("Number of questions:", len(data["questions"]))
print()

# Show the first 3 in a clean format
for i, q in enumerate(data["questions"][:3], start=1):
    print(f"Q{i} [{q['type']}]: {q['question']}")
    print(f"   Sample answer: {q['sample_answer']}\n")

## 5. Experimenting with parameters

The project asks us to vary parameters and observe the effect. The two most
important for this use case are **`temperature`** and **`max_tokens`**.

- **`temperature`** (0–2): lower = more focused/deterministic, higher = more
  diverse/creative (and eventually incoherent).
- **`max_tokens`**: caps the response length. Too low and the answer gets cut off
  (you'll see `finish_reason == "length"`).

### 5a. Effect of temperature

We ask the same question at three temperatures and compare. With a low
temperature the questions tend to be the "standard" ones; with a high
temperature we get more unusual / creative phrasings.

In [ ]:
role = "Data Analyst"

for temp in [0.0, 0.7, 1.4]:
    print(f"================  temperature = {temp}  ================")
    text = generate_interview_questions(role, temperature=temp, max_tokens=400)
    # Print just the first ~2 questions to keep the output short
    print(text.strip()[:600], "...\n")

### 5b. Effect of max_tokens (watching for truncation)

Here we deliberately set `max_tokens` low to see the response get cut off, then
inspect `finish_reason`.

In [ ]:
def generate_with_finish_reason(role, **params):
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Job role: {role}"},
        ],
        **params,
    )
    choice = completion.choices[0]
    return choice.message.content, choice.finish_reason


for mt in [60, 800]:
    text, reason = generate_with_finish_reason("Data Analyst", max_tokens=mt, temperature=0.5)
    print(f"--- max_tokens = {mt} | finish_reason = {reason} ---")
    print(text.strip()[:300], "\n")

### 5c. top_p (nucleus sampling)

`top_p` is an alternative way to control randomness. OpenAI recommends changing
**either** `temperature` **or** `top_p`, not both. A low `top_p` keeps only the
most probable words, giving safe/predictable output.

In [ ]:
for p in [0.1, 1.0]:
    print(f"================  top_p = {p}  ================")
    text = generate_interview_questions("Data Analyst", top_p=p, max_tokens=300)
    print(text.strip()[:400], "...\n")

## 6. Results & findings

Record what you actually observed when you ran the cells above. Typical findings
for this use case:

- **temperature = 0** gives the same, fairly generic question set every run —
  great for reproducibility, a bit boring.
- **temperature ≈ 0.7** was our sweet spot: varied, realistic questions that
  still stayed on-topic. We use this as the default in the app.
- **temperature = 1.4+** started producing odd or off-topic questions and
  inconsistent formatting.
- **max_tokens too low** truncated the answer mid-sentence (`finish_reason == "length"`),
  so for 10 Q&A pairs we needed a generous budget (~800-1200 tokens).
- **JSON mode** (`response_format`) made the output trivial to plug into the
  Streamlit app — no fragile text parsing.
- The **system prompt** did the heavy lifting: explicitly asking for "exactly 10",
  a mix of question types, and a bias-free instruction changed the output far
  more than any parameter tweak.

## 7. Best practices we applied

- **Secrets in `.env`**, loaded with `python-dotenv`; the key is never written in
  the notebook and `.env` is git-ignored.
- **One configurable `MODEL` variable** instead of hard-coding the model everywhere.
- **Reusable functions** with `**params` so experiments don't duplicate code.
- **Structured (JSON) output** for anything an app will consume.
- A **clear, constrained system prompt** (count, types, bias-awareness) — most of
  the quality comes from here.
- Checking **`finish_reason`** to detect truncated responses.

### Companion app
`interview_question_app.py` is a Streamlit front-end that reuses this exact prompt.
Run it with:

```bash
streamlit run interview_question_app.py
```